# 01 线性方程组与三角求解

本章研究线性方程组

$$
A\boldsymbol{x}=\boldsymbol{b}
$$

的直接求解方法。直接方法的核心不是显式计算 $A^{-1}$，而是通过消元、三角分解或正交分解把问题转化为更容易求解的三角方程组。

本节先建立三个基础概念：矩阵结构、前代/回代、残差与前向误差。


In [ ]:
import pathlib
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
CHAPTER = ROOT / "chapters" / "ch06_direct_linear_systems"
SCRIPTS = CHAPTER / "scripts"
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from py_sc import backward_substitution, forward_substitution
from py_sc.direct_linear import relative_forward_error, relative_residual


## 直接方法与矩阵结构

直接方法在精确算术中通常有限步完成；但在浮点运算中，“有限步”不等于“精确解”。主元选择、舍入误差和条件数都会影响结果。

不同矩阵结构适合不同方法：

| 矩阵类型 | 常用方法 | 原因 |
| --- | --- | --- |
| 一般稠密矩阵 | 高斯消元、PLU | 适用范围广 |
| 对称正定矩阵 | Cholesky、$LDL^T$ | 利用对称和正定结构 |
| 三对角矩阵 | Thomas 追赶法 | 时间和存储为 $O(n)$ |
| 最小二乘或正交稳定性需求 | QR | 正交变换不放大二范数 |


## 前代

对下三角系统

$$
L\boldsymbol{y}=\boldsymbol{b},
$$

第 $i$ 个未知量只依赖前面已经求出的未知量：

$$
y_i=\frac{b_i-\sum_{j=1}^{i-1}l_{ij}y_j}{l_{ii}}.
$$


In [ ]:
def teaching_forward_substitution(L, b):
    L = np.asarray(L, dtype=float)
    b = np.asarray(b, dtype=float)
    n = L.shape[0]
    y = np.zeros(n)
    for i in range(n):
        y[i] = (b[i] - L[i, :i] @ y[:i]) / L[i, i]
    return y

L = np.array([[2.0, 0.0, 0.0], [1.0, 3.0, 0.0], [-1.0, 2.0, 4.0]])
y_true = np.array([1.0, -2.0, 0.5])
b = L @ y_true
print("教学版前代:", teaching_forward_substitution(L, b))
print("正式实现前代:", forward_substitution(L, b))


## 回代

对上三角系统

$$
U\boldsymbol{x}=\boldsymbol{y},
$$

从最后一个未知量开始：

$$
x_i=\frac{y_i-\sum_{j=i+1}^{n}u_{ij}x_j}{u_{ii}},
\qquad i=n,n-1,\ldots,1.
$$


In [ ]:
def teaching_backward_substitution(U, y):
    U = np.asarray(U, dtype=float)
    y = np.asarray(y, dtype=float)
    n = U.shape[0]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i + 1:] @ x[i + 1:]) / U[i, i]
    return x

U = np.array([[2.0, -1.0, 3.0], [0.0, 4.0, 1.0], [0.0, 0.0, -2.0]])
x_true = np.array([1.0, -2.0, 0.5])
y = U @ x_true
print("教学版回代:", teaching_backward_substitution(U, y))
print("正式实现回代:", backward_substitution(U, y))


## 残差、前向误差和条件数

数值解 $\widehat{\boldsymbol{x}}$ 的残差是

$$
\boldsymbol{r}=\boldsymbol{b}-A\widehat{\boldsymbol{x}}.
$$

若已知精确解，可以计算前向误差

$$
\frac{\|\widehat{\boldsymbol{x}}-\boldsymbol{x}\|_2}{\|\boldsymbol{x}\|_2}.
$$

条件数刻画问题本身对扰动的敏感性。病态矩阵上，小残差不一定意味着小前向误差。


In [ ]:
def hilbert_matrix(n):
    i = np.arange(1, n + 1)
    return 1.0 / (i[:, None] + i[None, :] - 1.0)

n = 10
A = hilbert_matrix(n)
x_exact = np.ones(n)
b = A @ x_exact
x_computed = np.linalg.solve(A, b)
print("条件数:", np.linalg.cond(A))
print("相对残差:", relative_residual(A, x_computed, b))
print("相对前向误差:", relative_forward_error(x_computed, x_exact))


## 小结

* 直接法通常把原问题转化为一个或多个三角方程组。
* 前代和回代是高斯消元、LU、Cholesky、LDL^T 和 QR 求解的共同底层步骤。
* 评估线性系统求解结果时，应同时看残差、前向误差和条件数。
* 下一节开始讨论一般稠密矩阵上的高斯消元和主元选择。
